# IAM Handwriting Recognition Benchmark Analysis

## Executive Summary

**Dataset:** IAM Mini - Handwriting Recognition  
**Total Samples:** 500 handwritten text images per phase  
**Task:** Optical Character Recognition (OCR) of cursive and print handwriting  
**Evaluation Metrics:** CER (Character Error Rate), WER (Word Error Rate), Cosine Similarity

## Benchmark Structure

### Phase Pa: OCR Baseline (Pure OCR Models)
- **Models:** Azure Document Intelligence, Mistral Document AI
- **Approach:** Direct OCR on handwritten text
- **Purpose:** Establish baseline OCR performance for handwriting recognition

### Phase Pb: VLM Baseline (Generic Prompting)
- **Models:** GPT-5-mini, GPT-5-nano, Claude Sonnet  
- **Prompt:** Generic text extraction (no handwriting context)
- **Purpose:** Evaluate general-purpose VLM capabilities for handwriting OCR

### Phase Pc: VLM with Task-Aware Prompting
- **Models:** GPT-5-mini, GPT-5-nano, Claude Sonnet
- **Prompt:** Handwriting-specific instructions
- **Purpose:** Evaluate impact of task-aware prompting on handwriting recognition

## Key Metrics

- **CER (Character Error Rate):** Edit distance at character level (lower is better, 0.0 = perfect)
- **WER (Word Error Rate):** Edit distance at word level (lower is better, 0.0 = perfect)
- **Cosine Similarity:** Semantic similarity using embeddings (higher is better, 1.0 = identical)

## Analysis Focus Areas

1. **OCR vs VLM:** Do vision language models outperform specialized OCR for handwriting?
2. **Prompting Impact:** How much does task-aware prompting improve VLM performance?
3. **Writing Style:** Cursive vs print handwriting recognition challenges
4. **Model Comparison:** Trade-offs between speed and accuracy across models
5. **Error Patterns:** Character-level vs word-level error analysis

## To Run This Analysis

1. Ensure consolidated data exists in `../../2_clean/IAM_mini/`
2. This notebook will load Pa, Pb, Pc results and generate:
   - Character and word error rate calculations
   - Semantic similarity analysis
   - Model comparison visualizations
   - Sample-level analysis (easiest/hardest samples)
   - Detailed error analysis

## 1. Imports and Metadata

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sys
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Progress bar for long operations
from tqdm.notebook import tqdm

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

# Import evaluation metrics
from ocr_vs_vlm.metrics.evaluation_metrics import (
    calculate_cer,
    calculate_wer,
    compute_anls,
    compute_exact_match,
    compute_ground_truth_in_prediction
)

# Import embedding cache manager for efficient cosine similarity computation
from ocr_vs_vlm.metrics.embedding_cache import EmbeddingCacheManager

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

print("Libraries and evaluation metrics loaded successfully!")

Libraries and evaluation metrics loaded successfully!


## 2. Dataset Explorer

Load all phase files and explore the dataset structure.

In [2]:
# Define paths
RESULTS_DIR = Path("../../2_clean/IAM_mini")

# Check available files
available_files = list(RESULTS_DIR.glob("*.csv"))
print("Available files:")
for f in sorted(available_files):
    print(f"  - {f.name}")

Available files:
  - Pa.csv
  - Pb.csv
  - Pc.csv
  - phase_1.csv
  - phase_2.csv
  - phase_3.csv


In [3]:
# Dataset configuration
DATASET_NAME = "IAM_mini"

# Initialize embedding cache manager
# This will load any previously computed embeddings from 3_embeddings/
EMBEDDINGS_DIR = Path("../../3_embeddings")
embedding_manager = EmbeddingCacheManager(DATASET_NAME, EMBEDDINGS_DIR)

print(f"📁 Dataset: {DATASET_NAME}")
print(f"📂 Embeddings directory: {EMBEDDINGS_DIR.resolve()}")
if embedding_manager.cache:
    print(f"✅ Loaded cached embeddings for phases: {', '.join(embedding_manager.cache.keys())}")
else:
    print("⚠️ No cached embeddings found - will compute on first run and save for future use")

📁 Dataset: IAM_mini
📂 Embeddings directory: /Users/kenzabenkirane/Documents/GitHub/research-playground/ocr_vs_vlm/results/3_embeddings
⚠️ No cached embeddings found - will compute on first run and save for future use


In [4]:
# Load results for each phase
phase_dfs = {}

for phase in ['Pa', 'Pb', 'Pc']:
    file_path = RESULTS_DIR / f"{phase}.csv"
    if file_path.exists():
        phase_dfs[phase] = pd.read_csv(file_path)
        print(f"{phase}: {phase_dfs[phase].shape[0]} samples, {phase_dfs[phase].shape[1]} columns")
    else:
        print(f"{phase}: Not available")

print(f"\nTotal phases loaded: {len(phase_dfs)}")

Pa: 500 samples, 13 columns
Pb: 500 samples, 17 columns
Pc: 500 samples, 17 columns

Total phases loaded: 3


In [5]:
# Inspect column names for each phase
for phase, df in phase_dfs.items():
    print(f"\n{phase} columns:")
    pred_cols = [col for col in df.columns if col.startswith('prediction_')]
    models = [col.replace('prediction_', '') for col in pred_cols]
    print(f"  Models: {', '.join(models)}")
    print(f"  Ground truth column: {'ground_truth' if 'ground_truth' in df.columns else 'NOT FOUND'}")
    print(f"  Total columns: {len(df.columns)}")


Pa columns:
  Models: azure_intelligence, mistral_document_ai
  Ground truth column: ground_truth
  Total columns: 13

Pb columns:
  Models: claude_sonnet, gpt-5-mini, gpt-5-nano
  Ground truth column: ground_truth
  Total columns: 17

Pc columns:
  Models: claude_sonnet, gpt-5-mini, gpt-5-nano
  Ground truth column: ground_truth
  Total columns: 17


### Dataset Statistics

In [6]:
phase = 'Pa'

In [7]:
# Show basic statistics

if phase in phase_dfs:
    print(f"\nBasic statistics for phase {phase}:")
    df_base = phase_dfs[phase]
    
    print("Dataset Statistics:")
    print(f"  Total samples: {len(df_base)}")
    
    if 'ground_truth' in df_base.columns:
        # Text length statistics
        text_lengths = df_base['ground_truth'].str.len()
        print(f"\nGround Truth Text Length:")
        print(f"  Mean: {text_lengths.mean():.1f} characters")
        print(f"  Median: {text_lengths.median():.1f} characters")
        print(f"  Min: {text_lengths.min()} characters")
        print(f"  Max: {text_lengths.max()} characters")
        
        # Word count statistics
        word_counts = df_base['ground_truth'].str.split().str.len()
        print(f"\nGround Truth Word Count:")
        print(f"  Mean: {word_counts.mean():.1f} words")
        print(f"  Median: {word_counts.median():.1f} words")
        print(f"  Min: {word_counts.min()} words")
        print(f"  Max: {word_counts.max()} words")


Basic statistics for phase Pa:
Dataset Statistics:
  Total samples: 500

Ground Truth Text Length:
  Mean: 377.1 characters
  Median: 376.0 characters
  Min: 96 characters
  Max: 625 characters

Ground Truth Word Count:
  Mean: 72.6 words
  Median: 72.0 words
  Min: 19 words
  Max: 116 words


### Sample Predictions Preview

Display 10 random predictions from 3 different models to get a qualitative sense of performance.

In [8]:
# Get 10 random samples from Pa phase
if phase in phase_dfs:
    df_samples = phase_dfs[phase].sample(n=min(10, len(phase_dfs[phase])), random_state=42)
    
    # Get model names
    pred_cols = [col for col in df_samples.columns if col.startswith('prediction_')]
    models = [col.replace('prediction_', '') for col in pred_cols[:3]]  # First 3 models
    
    print("="*120)
    print("SAMPLE PREDICTIONS - Showing 10 random samples")
    print("="*120)
    
    for idx, row in df_samples.iterrows():
        print(f"\nSample: {row['sample_id']}")
        print(f"Ground Truth: \n{row['ground_truth']}")
        print("-" * 120)
        
        for model in models:
            pred_col = f'prediction_{model}'
            if pred_col in row:
                pred_text = str(row[pred_col])
                display_text = pred_text
                print(f"{model:30s}: \n{display_text}")
        print("=" * 120)

SAMPLE PREDICTIONS - Showing 10 random samples

Sample: iam_454_k07-063a
Ground Truth: 
He looked at her . Heard thrown back in a pool of hair ,
her blood-red lips parted and the beating of her
heart in the full throat . Her mouth did things he
thought no human being could stand without
dying , but he went on living in an ocean of
voluptuousness , that swelled and ebbed over him ,
under him , in him and through him ...
------------------------------------------------------------------------------------------------------------------------
azure_intelligence            : 
He looked at her. Head thrown back in a pool of hair, her blood-red lips parted and
the beating of her heart in the full throat. Her mouth did things he thought no human
being could stand without dying, but he went on living in an ocean of voluptuousness,
that swelled and ebbed over him, under him, in him and through him ...
mistral_document_ai           : 
He looked at her. Head thrown back in a pool of hair, her blood

## 3. Metrics Evaluation

Calculate CER, WER, and cosine similarity for all models across all samples.

In [ ]:
# Function to calculate metrics for a single prediction (using embedding cache)
def calculate_sample_metrics(
    ground_truth: str, 
    prediction: str,
    phase: str,
    sample_id: str,
    model: str,
    emb_manager: EmbeddingCacheManager
) -> Dict[str, float]:
    """Calculate all metrics for a single sample with cached embeddings."""
    if pd.isna(prediction) or prediction == "":
        return {
            'cer': 1.0,
            'wer': 1.0,
            'cosine_similarity': 0.0,
            'ground_truth_in_prediction': 0.0,
        }
    
    # Use embedding manager for cosine similarity (with caching)
    cosine_sim = emb_manager.compute_cosine_similarity(
        phase=phase,
        ground_truth=ground_truth,
        prediction=str(prediction),
        sample_id=sample_id,
        model=model
    )
    
    return {
        'cer': calculate_cer(ground_truth, prediction),
        'wer': calculate_wer(ground_truth, prediction),
        'cosine_similarity': cosine_sim,
        'ground_truth_in_prediction': compute_ground_truth_in_prediction(str(prediction), [ground_truth]),
    }

# Calculate metrics for all phases and models
metrics_results = {}

for phase, df in phase_dfs.items():
    print(f"\n📊 Calculating metrics for {phase}...")
    
    # Get all prediction columns
    pred_cols = [col for col in df.columns if col.startswith('prediction_')]
    
    phase_metrics = {}
    
    for pred_col in pred_cols:
        model = pred_col.replace('prediction_', '')
        print(f"   Processing model: {model}")
        
        # Calculate metrics for each sample with progress bar
        metrics_list = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"   {model}", leave=False):
            metrics = calculate_sample_metrics(
                ground_truth=row['ground_truth'],
                prediction=row[pred_col],
                phase=phase,
                sample_id=row['sample_id'],
                model=model,
                emb_manager=embedding_manager
            )
            metrics_list.append(metrics)
        
        # Aggregate metrics
        phase_metrics[model] = {
            'cer': np.mean([m['cer'] for m in metrics_list]),
            'wer': np.mean([m['wer'] for m in metrics_list]),
            'cosine_similarity': np.mean([m['cosine_similarity'] for m in metrics_list]),
            'cer_std': np.std([m['cer'] for m in metrics_list]),
            'wer_std': np.std([m['wer'] for m in metrics_list]),
            'cosine_similarity_std': np.std([m['cosine_similarity'] for m in metrics_list]),
            'ground_truth_in_prediction': np.mean([m['ground_truth_in_prediction'] for m in metrics_list]),
            'ground_truth_in_prediction_std': np.std([m['ground_truth_in_prediction'] for m in metrics_list])
        }
        
        print(f"   ✅ {model}: CER={phase_metrics[model]['cer']:.4f}, WER={phase_metrics[model]['wer']:.4f}, Cosine={phase_metrics[model]['cosine_similarity']:.4f}")
    
    metrics_results[phase] = phase_metrics

# Print cache statistics
embedding_manager.print_statistics()
print("\n✅ Metrics calculation complete!")


📊 Calculating metrics for Pa...
   Processing model: azure_intelligence


   azure_intelligence:   0%|          | 0/500 [00:00<?, ?it/s]

In [ ]:
# Create a summary DataFrame
summary_data = []

for phase, models in metrics_results.items():
    for model, metrics in models.items():
        summary_data.append({
            'Phase': phase,
            'Model': model,
            'CER': metrics['cer'],
            'WER': metrics['wer'],
            'Cosine Similarity': metrics['cosine_similarity']
            'Ground Truth in Prediction': metrics['ground_truth_in_prediction']
        })

summary_df = pd.DataFrame(summary_data)

print("\nMetrics Summary:")
print("=" * 100)
display(summary_df.sort_values(['Phase', 'CER']))

# Best model per phase
print("\nBest Model per Phase (by CER):")
for phase in summary_df['Phase'].unique():
    phase_data = summary_df[summary_df['Phase'] == phase]
    best_model = phase_data.loc[phase_data['CER'].idxmin()]
    print(f"  {phase}: {best_model['Model']} (CER={best_model['CER']:.4f})")

## 4. Overall View

Combined visualizations comparing all metrics across all models.

### Bar Chart Comparisons

In [ ]:
# Bar charts for each metric
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

metrics_to_plot = ['CER', 'WER', 'Cosine Similarity', 'Ground Truth in Prediction']
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#f39c12']

for idx, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    ax = axes[idx]
    
    # Pivot data for grouped bar chart
    pivot_data = summary_df.pivot(index='Model', columns='Phase', values=metric)
    
    pivot_data.plot(kind='bar', ax=ax, color=['#e74c3c', '#3498db', '#2ecc71'], alpha=0.8)
    
    ax.set_title(f'{metric} by Model and Phase', fontsize=14, fontweight='bold')
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel(metric, fontsize=12)
    ax.legend(title='Phase', title_fontsize=11, fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    # Add value labels on bars
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=8)

plt.tight_layout()
plt.show()

### Box Plot Distribution Analysis

In [ ]:
# Calculate per-sample metrics for box plots
sample_metrics_data = []

for phase, df in phase_dfs.items():
    pred_cols = [col for col in df.columns if col.startswith('prediction_')]
    
    for pred_col in pred_cols:
        model = pred_col.replace('prediction_', '')
        
        for _, row in df.iterrows():
            metrics = calculate_sample_metrics(row['ground_truth'], row[pred_col])
            
            sample_metrics_data.append({
                'Phase': phase,
                'Model': model,
                'CER': metrics['cer'],
                'WER': metrics['wer'],
                'Cosine Similarity': metrics['cosine_similarity']
                'Ground Truth in Prediction': metrics['ground_truth_in_prediction']
            })

sample_metrics_df = pd.DataFrame(sample_metrics_data)

# Create box plots
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for idx, metric in enumerate(['CER', 'WER', 'Cosine Similarity', 'Ground Truth in Prediction']):
    ax = axes[idx]
    
    # Create box plot
    sns.boxplot(data=sample_metrics_df, x='Model', y=metric, hue='Phase', ax=ax, palette='Set2')
    
    ax.set_title(f'{metric} Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Heatmap: Model vs Metric Performance

In [ ]:
# Create heatmaps for each phase
fig, axes = plt.subplots(1, len(metrics_results), figsize=(6 * len(metrics_results), 6))

if len(metrics_results) == 1:
    axes = [axes]

for idx, (phase, models) in enumerate(metrics_results.items()):
    ax = axes[idx]
    
    # Create heatmap data
    heatmap_data = []
    model_names = []
    
    for model, metrics in models.items():
        model_names.append(model)
        heatmap_data.append([
            metrics['cer'],
            metrics['wer'],
            metrics['cosine_similarity']
        ])
    
    heatmap_df = pd.DataFrame(
        heatmap_data,
        index=model_names,
        columns=['CER', 'WER', 'Cosine Similarity']
    )
    
    # Normalize for better visualization (lower is better for CER/WER, higher for cosine)
    heatmap_normalized = heatmap_df.copy()
    heatmap_normalized['CER'] = 1 - heatmap_normalized['CER']  # Invert so higher is better
    heatmap_normalized['WER'] = 1 - heatmap_normalized['WER']  # Invert so higher is better
    
    sns.heatmap(heatmap_normalized, annot=heatmap_df, fmt='.3f', cmap='RdYlGn', 
                ax=ax, cbar_kws={'label': 'Performance (higher is better)'}, 
                vmin=0, vmax=1, linewidths=0.5)
    
    ax.set_title(f'Phase {phase} - Model Performance Heatmap', fontsize=14, fontweight='bold')
    ax.set_xlabel('Metric', fontsize=12)
    ax.set_ylabel('Model', fontsize=12)

plt.tight_layout()
plt.show()

### Scatter Plot: CER vs WER Correlation

In [ ]:
# Scatter plot of CER vs WER for each phase
fig, axes = plt.subplots(1, len(metrics_results), figsize=(6 * len(metrics_results), 6))

if len(metrics_results) == 1:
    axes = [axes]

for idx, phase in enumerate(metrics_results.keys()):
    ax = axes[idx]
    
    phase_data = summary_df[summary_df['Phase'] == phase]
    
    scatter = ax.scatter(phase_data['CER'], phase_data['WER'], 
                        s=200, alpha=0.6, c=range(len(phase_data)), cmap='viridis')
    
    # Add model labels
    for _, row in phase_data.iterrows():
        ax.annotate(row['Model'], (row['CER'], row['WER']), 
                   fontsize=9, ha='right', va='bottom')
    
    ax.set_title(f'Phase {phase} - CER vs WER', fontsize=14, fontweight='bold')
    ax.set_xlabel('CER (Character Error Rate)', fontsize=12)
    ax.set_ylabel('WER (Word Error Rate)', fontsize=12)
    ax.grid(alpha=0.3)
    
    # Add diagonal reference line
    lims = [0, max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'r--', alpha=0.3, label='CER=WER')
    ax.legend()

plt.tight_layout()
plt.show()

### Error Analysis: Best and Worst Samples

In [ ]:
# Analyze best and worst performing samples for Pa phase
if 'Pa' in phase_dfs:
    df_analysis = phase_dfs['Pa'].copy()
    
    # Get first model for analysis
    pred_cols = [col for col in df_analysis.columns if col.startswith('prediction_')]
    first_model = pred_cols[0].replace('prediction_', '')
    
    # Calculate CER for each sample
    df_analysis['cer'] = df_analysis.apply(
        lambda row: calculate_cer(row['ground_truth'], row[f'prediction_{first_model}']),
        axis=1
    )
    
    # Get best and worst samples
    best_samples = df_analysis.nsmallest(5, 'cer')
    worst_samples = df_analysis.nlargest(5, 'cer')
    
    print("\n" + "="*120)
    print(f"BEST PERFORMING SAMPLES (Lowest CER) - Model: {first_model}")
    print("="*120)
    
    for _, row in best_samples.iterrows():
        print(f"\nSample: {row['sample_id']} | CER: {row['cer']:.4f}")
        print(f"Ground Truth: {row['ground_truth'][:100]}..." if len(str(row['ground_truth'])) > 100 else f"Ground Truth: {row['ground_truth']}")
        pred_text = str(row[f'prediction_{first_model}'])
        print(f"Prediction:   {pred_text[:100]}..." if len(pred_text) > 100 else f"Prediction:   {pred_text}")
    
    print("\n" + "="*120)
    print(f"WORST PERFORMING SAMPLES (Highest CER) - Model: {first_model}")
    print("="*120)
    
    for _, row in worst_samples.iterrows():
        print(f"\nSample: {row['sample_id']} | CER: {row['cer']:.4f}")
        print(f"Ground Truth: {row['ground_truth'][:100]}..." if len(str(row['ground_truth'])) > 100 else f"Ground Truth: {row['ground_truth']}")
        pred_text = str(row[f'prediction_{first_model}'])
        print(f"Prediction:   {pred_text[:100]}..." if len(pred_text) > 100 else f"Prediction:   {pred_text}")

### Inference Time Analysis

In [ ]:
# Analyze inference times across models and phases
inference_data = []

for phase, df in phase_dfs.items():
    time_cols = [col for col in df.columns if col.startswith('inference_time_ms_')]
    
    for time_col in time_cols:
        model = time_col.replace('inference_time_ms_', '')
        
        mean_time = df[time_col].mean()
        median_time = df[time_col].median()
        
        inference_data.append({
            'Phase': phase,
            'Model': model,
            'Mean Time (ms)': mean_time,
            'Median Time (ms)': median_time
        })

if inference_data:
    inference_df = pd.DataFrame(inference_data)
    
    print("\nInference Time Summary:")
    print("="*100)
    display(inference_df.sort_values(['Phase', 'Mean Time (ms)']))
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    
    pivot_time = inference_df.pivot(index='Model', columns='Phase', values='Mean Time (ms)')
    pivot_time.plot(kind='bar', ax=ax, color=['#e74c3c', '#3498db', '#2ecc71'], alpha=0.8)
    
    ax.set_title('Average Inference Time by Model and Phase', fontsize=14, fontweight='bold')
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Time (ms)', fontsize=12)
    ax.legend(title='Phase', title_fontsize=11, fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
else:
    print("No inference time data available")

## 5. LLM Query Section

This section is a placeholder for analyzing the notebook outputs using an LLM.

### Key Questions to Investigate:

1. **Performance Comparison:**
   - Which model performs best overall for handwriting recognition?
   - How do OCR models (Pa) compare to VLMs (Pb, Pc)?
   - What is the impact of task-aware prompting (Pc vs Pb)?

2. **Metric Insights:**
   - Is there a strong correlation between CER and WER?
   - Do models with low CER/WER also have high cosine similarity?
   - Are there models that excel at one metric but not others?

3. **Error Patterns:**
   - What types of handwriting cause the most errors?
   - Do models struggle more with cursive or print text?
   - Are there common character substitution patterns?

4. **Speed vs Accuracy:**
   - What is the trade-off between inference time and accuracy?
   - Which model offers the best balance?

5. **Recommendations:**
   - Which model should be used for production handwriting OCR?
   - Are specialized OCR models still necessary, or can VLMs replace them?
   - What improvements could be made to prompting strategies?

In [ ]:
# Placeholder for LLM analysis
# This cell can be used to:
# 1. Export summary statistics to a text file
# 2. Generate natural language insights
# 3. Create a final report

print("LLM Query Section - Ready for analysis")
print("\nAvailable data for LLM analysis:")
print(f"  - Summary metrics: {len(summary_df)} rows")
print(f"  - Sample-level metrics: {len(sample_metrics_df)} rows")
print(f"  - Inference time data: {len(inference_df) if 'inference_df' in locals() else 0} rows")
print("\nNext steps:")
print("  1. Export key statistics to JSON/text for LLM ingestion")
print("  2. Ask LLM to analyze patterns and generate insights")
print("  3. Create actionable recommendations based on findings")

## 6. Save Embeddings Cache

Save any newly computed embeddings to disk for faster future runs.

In [ ]:
# Save any newly computed embeddings to disk for future use
if embedding_manager.modified_phases:
    print("💾 Saving newly computed embeddings...")
    saved_files = embedding_manager.save_new_embeddings()
    for f in saved_files:
        print(f"   ✅ Saved: {f}")
    print(f"\n📁 Embeddings saved to {EMBEDDINGS_DIR.resolve()}")
    print("   Next run will load these embeddings from cache (much faster!)")
else:
    print("✨ All embeddings were loaded from cache - no new embeddings to save.")

## Conclusion

This notebook provides a comprehensive analysis of handwriting recognition performance across OCR and VLM models.

**Key Deliverables:**
- Quantitative metrics (CER, WER, Cosine Similarity) for all models
- Visual comparisons across phases and models
- Error analysis identifying strengths and weaknesses
- Inference time benchmarks

**Next Steps:**
- Use LLM to generate natural language insights
- Identify specific handwriting styles that cause issues
- Recommend optimal model selection strategy